In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_data = pd.read_csv('../Train.csv')
print("Shape of train_data: ", train_data.shape)

# Input
X = train_data.iloc[:, 1:].values # All item in row, from 2nd index to last
# Y true
y = train_data.iloc[:, 0].values # All item in row, first index only

print("Shape fo x after separating features:", X.shape)


Shape of train_data:  (42000, 785)
Shape fo x after separating features: (42000, 784)


In [57]:
def one_hot_encode(y_true, num_classes=10):
    one_hot = np.zeros((y_true.shape[0], num_classes))
    one_hot[np.arange(y_true.shape[0]), y_true] = 1
    return one_hot

In [58]:
# One-Hot encode the labels
y = one_hot_encode(y, 10)
print("One-hot encoded y: ", y.shape)
# https://stackoverflow.com/questions/49054538/how-to-split-the-data-set-without-train-test-split
# Split the data for training and testing
train_pct_index = int(0.8 * len(X))
X_train, X_test = X[:train_pct_index], X[train_pct_index:]
y_train, y_test = y[:train_pct_index], y[train_pct_index:]

# Shuffle the training data
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]
print(X_train.shape)

One-hot encoded y:  (42000, 10)
(33600, 784)


In [59]:
# Initialize weights and biases
# Self preference kung anong size piliin sa hidden layer
W1 = np.random.randn(784, 256) * np.sqrt(1.0 / 784)
B1 = np.zeros((1, 256))

W2 = np.random.randn(256, 128) * np.sqrt(1.0 / 256)
B2 = np.zeros((1, 128))

W3 = np.random.randn(128, 10) * np.sqrt(1.0 / 128)
B3 = np.zeros((1, 10))

print(W1.shape)

(784, 256)


In [60]:
def update_params(W1, W2, W3, B1, B2, B3, gradients, eta):
    w3, b3, w2, b2, w1, b1 = gradients
    W3 -= w3*eta
    B3 -= b3*eta
    W2 -= w2*eta
    B2 -= b2*eta
    W1 -= w1*eta
    B1 -= b1*eta

    return W1, W2, W3, B1, B2, B3


In [61]:
def relu(x_array):
    return np.maximum(0, x_array)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(x):
    return 1.0/(1.0 + np.exp(-x))

def sigmoid_derivative(sigmoid_output):
    return sigmoid_output * (1 - sigmoid_output)

def softmax(Z):
    exp_scores = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

def softmax_deriv_with_loss(A, y):
    return A - y

def cross_entropy(y, y_hat, batch_size):
    return -np.sum(y * np.log(y_hat + 1e-8)) / batch_size

def loss_derivative(y, y_hat):
    return y_hat - y

In [62]:
def forward_pass(X_train):
    # X_train @ W1 + B1
    Z1 = X_train @ W1 + B1
    # Applying sigmoid na muna kasi maya nayang ReLu HAHAHAHA sigmoid lang napagaralan ko eh
    H1 = sigmoid(Z1)
    # H1 @ W2 + B2
    Z2 = H1 @ W2 + B2
    #  Apply sigmoid ulit
    H2 = sigmoid(Z2)
    # H2 @ W3 + B3
    Z3 = H2 @ W3  + B3
    # Y pred
    H3 = softmax(Z3)

    return H1, H2, H3


In [63]:

""" First of all, PUTANGINANG CHAIN RULE """
def backprop(W2, W3, H1, H2, y_pred, y):
    """dL_dY_hat * dY_hat_dZ3 - small change in Z3 affects the loss"""
    dL_dZ3 = softmax_deriv_with_loss(y_pred, y)
    """ dL_dZ3 * dZ3_dW3 - small change in W3 affects the loss
     dZ3_dW3 = H2 * W3 = H2 """
    dL_dW3 = H2.T @ dL_dZ3 / X_train.shape[0]
    dL_db3 = np.sum(dL_dZ3, axis=0, keepdims=True) / X_train.shape[0]
    """
    find deriv at the hidden neuron H2:
    Z3 = H2 * W3, then dZ3_dH2 = W3

    to compute for the dL_dZ2 = dL_dZ3 * dZ3_dH2 * dH2_dZ2

    dH2_dZ2 = sigmoid_deriv
    """
    dL_dZ2 = (dL_dZ3 @ W3.T) * sigmoid_derivative(H2)
    """
    Z2 = W2 * H1, so dZ2_dW2 = H1

    dL_dW2 = dL_dZ2 * dZ2_dW2
    """
    dL_dW2 = H1.T @ dL_dZ2 / X_train.shape[0]
    dL_db2 = np.sum(dL_dZ2, axis=0, keepdims=True) / X_train.shape[0]
    """
    to find dL_dZ1 = dH1_dZ1 * dZ2_dH1 * dL_dZ2

    Z2 = H1 * W2, dZ2_dH1 = W2
    """
    dL_dZ1 = (dL_dZ2 @ W2.T) * sigmoid_derivative(H1)
    """
    Z1 = X_train * W1, so dZ1_dW1 = X_train
    dL_dW1 = dL_dZ1 * dZ1_dW1
    """
    dL_dW1 = X_train.T @ dL_dZ1 / X_train.shape[0]
    dL_db1 = np.sum(dL_dZ1, axis=0, keepdims=True) / X_train.shape[0]

    return dL_dW3, dL_db3, dL_dW2, dL_db2, dL_dW1, dL_db1

In [64]:
eta = 0.1
epochs = 1000

for e in range(epochs):
    H1, H2, y_hat = forward_pass(X_train)

    error = cross_entropy(y_train, y_hat, 33600)

    gradients = backprop(W2, W3, H1, H2, y_hat, y_train)

    W1, W2, W3, B1, B2, B3 = update_params(W1, W2, W3, B1, B2, B3, gradients, eta)

    pred_classes = np.argmax(y_hat, axis=1)
    true_classes = np.argmax(y_train, axis=1)
    accuracy = np.mean(pred_classes == true_classes)
    print(f"Epoch {e+1}/{epochs} - "
                  f"Loss: {error:.4f} - "
                  f"Train Acc: {accuracy:.4f} - "
        )



Epoch 1/1000 - Loss: 2.3369 - Train Acc: 0.1217 - 
Epoch 2/1000 - Loss: 2.2988 - Train Acc: 0.1439 - 
Epoch 3/1000 - Loss: 2.2720 - Train Acc: 0.1959 - 
Epoch 4/1000 - Loss: 2.2511 - Train Acc: 0.2503 - 
Epoch 5/1000 - Loss: 2.2334 - Train Acc: 0.2992 - 
Epoch 6/1000 - Loss: 2.2172 - Train Acc: 0.3446 - 
Epoch 7/1000 - Loss: 2.2015 - Train Acc: 0.3903 - 
Epoch 8/1000 - Loss: 2.1863 - Train Acc: 0.4286 - 
Epoch 9/1000 - Loss: 2.1717 - Train Acc: 0.4576 - 
Epoch 10/1000 - Loss: 2.1572 - Train Acc: 0.4813 - 
Epoch 11/1000 - Loss: 2.1430 - Train Acc: 0.5023 - 
Epoch 12/1000 - Loss: 2.1289 - Train Acc: 0.5204 - 
Epoch 13/1000 - Loss: 2.1148 - Train Acc: 0.5365 - 
Epoch 14/1000 - Loss: 2.1007 - Train Acc: 0.5508 - 
Epoch 15/1000 - Loss: 2.0865 - Train Acc: 0.5629 - 
Epoch 16/1000 - Loss: 2.0723 - Train Acc: 0.5742 - 
Epoch 17/1000 - Loss: 2.0583 - Train Acc: 0.5839 - 
Epoch 18/1000 - Loss: 2.0444 - Train Acc: 0.5923 - 
Epoch 19/1000 - Loss: 2.0303 - Train Acc: 0.6008 - 
Epoch 20/1000 - Loss: